# ICS 591 Team 2 — Evaluation 3: the **confidence threshold** for handing a photo to a supervisor

**How the live system uses the model** (backend `domain/verification.py`):
- model says **accept** with confidence ≥ **A** → household is paid automatically
- model says **reject** (dirty or wrong type) with confidence ≥ **R** → rejected automatically
- model says **retake** → the rider is asked for a new photo (no threshold)
- **everything else → a person decides** (rider / supervisor)

Today **A = 0.90 and R = 0.80**. This notebook asks: *what do those numbers give us, and would other numbers be better?*

For every threshold we count, per 100 pickups: how many are decided automatically, how many go to a person,
how many **wrong payments** slip through, and how many **honest households are paid without waiting**.

**Honest method.** Choosing thresholds on the same photos you report on flatters the result. So:
1. Part 1 reports the **current** setting on the test set (nothing is tuned).
2. Part 2 shows a sweep on the test set, **for understanding only**.
3. Part 3 **chooses** thresholds on the local *validation* photos (the 20% of training labels held out in notebook 05)
   and then reports them once on the test set.

**How to run:** Runtime → Run all. No GPU needed.

In [ ]:
# 1. Setup: paths, the local TEST set, helpers. A GPU is optional (113 images also run on CPU in a few minutes).
!pip -q install timm
import os, json, shutil, pathlib, random, hashlib
import numpy as np, pandas as pd, torch, timm, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from torchvision import transforms as T
from google.colab import drive
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive')
LOCAL_OUT = DRIVE / 'ics591_local_data'             # Stage 2a/2b: images + label files
IMG_DRIVE = LOCAL_OUT / 'raw' / 'images'
S1_OUT = DRIVE / 'ics591_model_selection' / 'realwaste'   # Stage 1 outputs
STAGE1_CKPT = S1_OUT / 'ViT-S_16.pt'                 # baseline: Stage 1 ViT, RealWaste only
FT_OUT = DRIVE / 'ics591_finetune'                   # Stage 2c outputs
V1_CKPT, V1_CFG = FT_OUT / 'verifier_v1.pt', FT_OUT / 'verifier_v1_config.json'
EVAL_OUT = DRIVE / 'ics591_evaluation'; EVAL_OUT.mkdir(exist_ok=True)   # everything this notebook saves goes here
TIMM_ID = 'vit_small_patch16_224.augreg_in21k_ft_in1k'
for p in [IMG_DRIVE, LOCAL_OUT / 'al_queue.csv']: assert p.exists(), f'missing: {p}'

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Running on', DEV)

MATERIAL = ['plastic', 'non_plastic', 'discard']
CLEAN = ['clean', 'contaminated']
DECISIONS = ['accept', 'reject_contaminated', 'reject_wrong_type', 'retake_photo']
NICE = {'accept': 'Accept', 'reject_contaminated': 'Reject: dirty', 'reject_wrong_type': 'Reject: wrong type',
        'retake_photo': 'Retake'}

# --- label files (exactly the same logic as notebook 05, so the test set is the same 113 images)
TEAM = ['nana', 'kelvin', 'cajetan']
QUEUE, AUTO = LOCAL_OUT / 'al_queue.csv', LOCAL_OUT / 'al_auto_labels.csv'
def REV(p): return LOCAL_OUT / f'al_reviews_{p}.csv'
def to_model_label(l): return 'discard' if str(l).startswith('discard') else l
def load_reviews():
    parts = [pd.read_csv(REV(p)) for p in TEAM if REV(p).exists()]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=['file', 'label', 'pool', 'labeler', 'ts'])
def test_files():
    q = pd.read_csv(QUEUE); return set(q.loc[q['pool'] == 'test', 'file'])
def test_labels():
    r = load_reviews(); r = r[r['pool'] == 'test'].sort_values('ts').drop_duplicates(['file', 'labeler'], keep='last')
    out = []
    for f, g in r.groupby('file'):
        vc = g['label'].map(to_model_label).value_counts()
        if len(vc) == 1 or vc.iloc[0] > vc.iloc[1]: out.append({'file': f, 'label': vc.index[0]})
    return pd.DataFrame(out, columns=['file', 'label'])
def current_labels():
    r = load_reviews(); r = r[r['pool'] != 'test'].sort_values('ts').drop_duplicates('file', keep='last')
    human = pd.DataFrame({'file': r['file'], 'label': r['label'].map(to_model_label), 'source': 'human'})
    auto = pd.read_csv(AUTO) if AUTO.exists() else pd.DataFrame(columns=['file', 'label'])
    auto = pd.DataFrame({'file': auto['file'], 'label': auto['label'], 'source': 'model'})
    auto = auto[~auto['file'].isin(human['file'])]
    lab = pd.concat([human, auto], ignore_index=True)
    return lab[~lab['file'].isin(test_files())].reset_index(drop=True)
LABEL_TO_DECISION = {'plastic_clean': 0, 'plastic_contaminated': 1, 'non_plastic': 2, 'discard': 3}
LABEL_TO_MATERIAL = {'plastic_clean': 0, 'plastic_contaminated': 0, 'non_plastic': 1, 'discard': 2}

loc_te = test_labels().sort_values('file').reset_index(drop=True)
print('Local TEST set:', len(loc_te), 'images', loc_te['label'].value_counts().to_dict())
if len(loc_te) != 113: print('WARNING: expected 113 test images (as in Stage 2c). Have label files changed since?')
y_dec = loc_te['label'].map(LABEL_TO_DECISION).values     # true decision: 0 accept, 1 dirty, 2 wrong type, 3 retake
y_mat = loc_te['label'].map(LABEL_TO_MATERIAL).values     # true material: 0 plastic, 1 non-plastic, 2 discard

CACHE = pathlib.Path('/content/img_cache'); CACHE.mkdir(exist_ok=True)
for f in loc_te['file']:
    if not (CACHE / f).exists(): shutil.copy(IMG_DRIVE / f, CACHE / f)

_cfg = timm.data.resolve_data_config({}, model=timm.create_model(TIMM_ID, pretrained=False))
EVAL_TF = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(_cfg['mean'], _cfg['std'])])

@torch.no_grad()
def run(model, files, batch=32):
    """Runs a model over image files; returns a list with one numpy array per model output."""
    model.eval(); outs = None
    for i in range(0, len(files), batch):
        x = torch.stack([EVAL_TF(Image.open(CACHE / f).convert('RGB')) for f in files[i:i + batch]]).to(DEV)
        o = model(x); o = o if isinstance(o, tuple) else (o,)
        o = [t.float().softmax(1).cpu().numpy() for t in o]
        outs = o if outs is None else [np.vstack([a, b]) for a, b in zip(outs, o)]
    return outs

def ci(correct, n_boot=2000):
    """95% bootstrap interval for a mean of 0/1 values (resample the test images 2000 times)."""
    v = np.asarray(correct, dtype=float); rng = np.random.default_rng(SEED)
    b = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(n_boot)]
    return [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]
def pct(x): return f'{100 * x:.1f}%'
def show_examples(idx, title, captions, path, cols=5):
    """Picture grid for slides."""
    idx = list(idx)[:15]
    if not idx: print('no examples'); return
    rows = int(np.ceil(len(idx) / cols)); fig, axs = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.6 * rows), squeeze=False)
    for a in axs.flat: a.axis('off')
    for a, i in zip(axs.flat, idx):
        a.imshow(Image.open(CACHE / loc_te['file'][i]).convert('RGB')); a.set_title(captions[i], fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path, dpi=130); plt.show()

In [ ]:
# Load the fine-tuned model v1 (the same weights that were converted to ONNX and deployed)
class VerifierV1(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone; d = backbone.num_features
        self.material = nn.Linear(d, len(MATERIAL)); self.clean = nn.Linear(d, len(CLEAN))
    def forward(self, x):
        f = self.backbone(x); return self.material(f), self.clean(f)

cfg = json.loads(V1_CFG.read_text())
bb = timm.create_model(TIMM_ID, pretrained=False, num_classes=0)
v1 = VerifierV1(bb); v1.load_state_dict(torch.load(V1_CKPT, map_location='cpu')); v1 = v1.to(DEV)
T_MAT, T_CLEAN = cfg['temperature_material'], cfg['temperature_clean']
print('weights sha256:', hashlib.sha256(V1_CKPT.read_bytes()).hexdigest()[:16], '| temperatures', round(T_MAT, 3), round(T_CLEAN, 3),
      '| classifier threshold in config', round(cfg['auto_decision_threshold'], 3))

def apply_T(p, t):
    z = np.log(np.clip(p, 1e-12, 1)) / t; z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def decide(pm, pc):
    """Same rule as notebook 05 and the deployed service (decision.py)."""
    mat = pm.argmax(1); cl = pc.argmax(1)
    dec = np.select([mat == 2, mat == 1, (mat == 0) & (cl == 1)], [3, 2, 1], default=0)
    conf = np.where(mat == 0, pm.max(1) * pc.max(1), pm.max(1))
    return dec, conf
def v1_predict(files):
    pm, pc = run(v1, list(files)); pm, pc = apply_T(pm, T_MAT), apply_T(pc, T_CLEAN)
    dec, conf = decide(pm, pc); return dec, conf, pm, pc

dec, conf, pm, pc = v1_predict(loc_te['file'])

# local validation split, recreated exactly as in notebook 05 (same labels, same seed)
from sklearn.model_selection import train_test_split
loc = current_labels()
_, loc_va = train_test_split(loc, test_size=0.2, stratify=loc['label'], random_state=SEED)
loc_va = loc_va.reset_index(drop=True)
for f in loc_va['file']:
    if not (CACHE / f).exists(): shutil.copy(IMG_DRIVE / f, CACHE / f)
dec_va, conf_va, _, _ = v1_predict(loc_va['file'])
y_va = loc_va['label'].map(LABEL_TO_DECISION).values
print('test photos', len(dec), '| validation photos', len(dec_va), loc_va['label'].value_counts().to_dict())

In [ ]:
# The policy, written once and used everywhere below
def simulate(A, R, dec, conf, y):
    """Apply the Engine's policy at thresholds A (accept) and R (reject). Returns per-100-pickup counts."""
    auto_acc = (dec == 0) & (conf >= A)
    auto_rej = ((dec == 1) | (dec == 2)) & (conf >= R)
    retake = dec == 3
    human = ~(auto_acc | auto_rej | retake)
    auto = auto_acc | auto_rej
    n = len(y); per100 = 100 / n
    return {
        'A': A, 'R': R,
        'paid_automatically': auto_acc.sum() * per100,
        'rejected_automatically': auto_rej.sum() * per100,
        'asked_to_retake': retake.sum() * per100,
        'to_a_person': human.sum() * per100,
        'wrong_payments': (auto_acc & (y != 0)).sum() * per100,              # money lost
        'honest_paid_without_waiting': (auto_acc & (y == 0)).sum() * per100,
        'honest_auto_rejected': (auto_rej & (y == 0)).sum() * per100,        # angry household, fixable on appeal
        'honest_asked_to_retake': (retake & (y == 0)).sum() * per100,
        'auto_money_correct': float(((auto_acc & (y == 0)) | (auto_rej & (y != 0))).sum() / max(1, auto.sum())),
        'auto_exact_correct': float((dec[auto] == y[auto]).mean()) if auto.any() else float('nan'),
    }
COLS = {'to_a_person': 'To a person', 'paid_automatically': 'Paid auto', 'rejected_automatically': 'Rejected auto',
        'asked_to_retake': 'Retake', 'wrong_payments': 'Wrong payments', 'honest_paid_without_waiting': 'Honest paid at once',
        'honest_auto_rejected': 'Honest auto-rejected', 'auto_money_correct': 'Auto decisions: money right',
        'auto_exact_correct': 'Auto decisions: exact reason right'}
def tidy(rows):
    d = pd.DataFrame(rows)
    for c in COLS:
        if c.startswith('auto_'): d[c] = (100 * d[c]).round(1).astype(str) + '%'
        else: d[c] = d[c].round(1)
    return d[['A', 'R', *COLS]].rename(columns=COLS)

In [ ]:
# PART 1. The current setting (A = 0.90, R = 0.80) on the test set. Numbers are per 100 pickups.
cur = simulate(0.90, 0.80, dec, conf, y_dec)
compare = [simulate(0.0, 0.0, dec, conf, y_dec), cur, simulate(1.01, 1.01, dec, conf, y_dec)]
t1 = tidy(compare); t1.insert(0, 'Setting', ['No threshold (model decides all)', 'Current: A 0.90 / R 0.80', 'Everything to a person (retakes still asked)'])
t1.to_csv(EVAL_OUT / 'threshold_current_setting.csv', index=False)
print(f"With the current setting, out of 100 pickups: {cur['to_a_person']:.0f} go to a person, "
      f"{cur['paid_automatically']:.0f} are paid automatically ({cur['wrong_payments']:.1f} of them wrongly), "
      f"{cur['rejected_automatically']:.0f} are rejected automatically and {cur['asked_to_retake']:.0f} are asked to retake.")
t1

In [ ]:
# PART 2. Sweep one threshold (A = R = t) on the test set -- to SEE the trade-off, not to choose
ts = np.round(np.arange(0.50, 1.0, 0.01), 2)
sw = pd.DataFrame([simulate(t, t, dec, conf, y_dec) for t in ts])
sw.to_csv(EVAL_OUT / 'threshold_sweep_test.csv', index=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(ts, sw['to_a_person'], 'k-', label='sent to a person')
ax[0].plot(ts, sw['honest_paid_without_waiting'], 'k--', label='honest households paid at once')
ax[0].plot(ts, sw['wrong_payments'], 'k:', lw=2, label='wrong payments')
ax[0].axvline(0.90, color='grey', lw=1); ax[0].text(0.902, 2, 'current A', fontsize=8, color='grey')
ax[0].set_xlabel('threshold t (A = R = t)'); ax[0].set_ylabel('per 100 pickups'); ax[0].legend(fontsize=8)
ax[0].set_title('More caution = more work for people')
ax[1].plot(sw['to_a_person'], 100 * sw['auto_money_correct'], 'k-', label='money decision right')
ax[1].plot(sw['to_a_person'], 100 * sw['auto_exact_correct'], 'k--', label='exact reason right')
ax[1].set_xlabel('photos sent to a person (per 100)'); ax[1].set_ylabel('% correct among automatic decisions')
ax[1].set_title('Accuracy of what the model decides alone'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(EVAL_OUT / 'threshold_sweep.png', dpi=150); plt.show()
tidy([simulate(t, t, dec, conf, y_dec) for t in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99]])

In [ ]:
# PART 3. CHOOSE thresholds on validation, then report once on test.
# Rule (written before looking): no wrong payments on validation, and at least 90% of automatic decisions money-right;
# among those, pick the setting that sends the FEWEST photos to a person.
grid = np.round(np.arange(0.50, 1.0, 0.02), 2)
cands = [simulate(A, R, dec_va, conf_va, y_va) for A in grid for R in grid]
ok = [c for c in cands if c['wrong_payments'] == 0 and c['auto_money_correct'] >= 0.90 and
      (c['paid_automatically'] + c['rejected_automatically']) > 0]
if ok:
    best = min(ok, key=lambda c: (c['to_a_person'], -c['A'], -c['R']))
    A_star, R_star = best['A'], best['R']
    print(f'Chosen on validation ({len(dec_va)} photos): A = {A_star}, R = {R_star}')
else:
    A_star, R_star = 0.90, 0.80
    print('No setting met the rule on validation -- keeping the current A 0.90 / R 0.80')
rows = [simulate(0.90, 0.80, dec, conf, y_dec), simulate(A_star, R_star, dec, conf, y_dec)]
t3 = tidy(rows); t3.insert(0, 'Setting', ['Current', 'Chosen on validation'])
t3.to_csv(EVAL_OUT / 'threshold_chosen_vs_current.csv', index=False)
json.dump({'current': {'A': 0.90, 'R': 0.80}, 'chosen_on_validation': {'A': A_star, 'R': R_star},
           'validation_n': int(len(dec_va)), 'test_n': int(len(dec)),
           'test_results': {k: (float(v) if isinstance(v, (int, float, np.floating)) else v) for k, v in rows[1].items()}},
          open(EVAL_OUT / 'threshold_results.json', 'w'), indent=2)
print('Test set results (per 100 pickups):'); t3

In [ ]:
# PART 4. Where do the confident mistakes sit? Confidence of each test photo, by whether the model was right
right = dec == y_dec
plt.figure(figsize=(9, 3.6))
for k, (m, mk) in enumerate([(right, 'o'), (~right, 'x')]):
    plt.scatter(conf[m], np.full(m.sum(), k) + np.random.default_rng(1).uniform(-.15, .15, m.sum()), marker=mk, c='k', s=25)
wp = (dec == 0) & (y_dec != 0)
plt.scatter(conf[wp], np.ones(wp.sum()) + 0.3, marker='v', c='k', s=40, label='would be a wrong payment')
plt.axvline(0.90, color='grey', ls='--', lw=1); plt.axvline(0.80, color='grey', ls=':', lw=1)
plt.yticks([0, 1], ['right', 'wrong']); plt.xlabel('model confidence'); plt.legend(fontsize=8, loc='lower left')
plt.title('Each test photo: dashed line = accept threshold 0.90, dotted = reject threshold 0.80')
plt.tight_layout(); plt.savefig(EVAL_OUT / 'threshold_confidence_strip.png', dpi=150); plt.show()
print('Wrong payments and their confidence:', sorted(np.round(conf[wp], 3).tolist()))

## How to read this
- **Part 1** is the answer for the slides: with today's thresholds, out of 100 pickups, how many does a person handle,
  and how many wrong payments still slip through?
- **Part 2** shows the trade-off: raising the threshold sends more photos to people but lets fewer wrong payments through.
- **Part 3** is the honest recommendation, chosen on validation and checked once on test. The validation set is small,
  so treat it as a starting point for the pilot, not a final number.
- **Part 4** shows whether the wrong payments sit above or below the 0.90 line. If they are all below it, the threshold is doing its job.
- Files are in `MyDrive/ics591_evaluation/`.